In [0]:
source_dir = "/Volumes/incremental_load/default/orders_data/source/"
target_dir = "/Volumes/incremental_load/default/orders_data/Archive/"
stage_table = "incremental_load.default.orders_stage"

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, DateType, TimestampType

schema = StructType([
    StructField("order_num", StringType(), True),
    StructField("tracking_num", StringType(), True),
    StructField("pck_recieved_date", DateType(), True),
    StructField("package_deliver_date", DateType(), True),
    StructField("status", StringType(), True),
    StructField("address", StringType(), True),
    StructField("last_update_timestamp", TimestampType(), True)
])


In [0]:
df = spark.read \
    .format("csv") \
    .option("header", "true") \
    .schema(schema) \
    .load(source_dir)


In [0]:
display(df)

In [0]:
# Create Delta table named stage_zn if it doesn't exist and overwrite the data in stage table
df.write.format("delta").mode("overwrite").saveAsTable(stage_table)

In [0]:
%sql
select * from incremental_load.default.orders_stage

In [0]:
# List all files in the source directory
files = dbutils.fs.ls(source_dir)

# Iterate on the list one by one and print each file path separately
for file in files:
    src_path = file.path

    # Construct the target path
    target_path = target_dir + src_path.split("/")[-1]
    
    # Move the file
    dbutils.fs.mv(src_path, target_path)